## Firewalls and troubleshooting

**The firewall** decides which packets the kernel accepts or drops. The filter lives in the kernel (**`nftables`**, superseding `iptables`); you drive it through a friendly front-end — **`ufw`** on Debian/Ubuntu, **`firewalld`** on RHEL/Fedora.

**`ufw`** is default-deny incoming, default-allow outgoing:

```bash
sudo ufw allow ssh          # allow port 22
sudo ufw allow 80/tcp
sudo ufw enable
```

⚠️ **Allow SSH *before* you `enable`** on a remote box, or you lock yourself out — the classic cloud-VM mistake. **`firewalld`** organises rules into **zones** per interface, with a two-step model — changes are runtime until you add `--permanent` then `--reload`:

```bash
sudo firewall-cmd --add-service=ssh --permanent
sudo firewall-cmd --reload
```

**Troubleshooting — work up the layers** when the network "isn't working":

1. **Connected?** `ip -br addr` — do I have an IP?
2. **Gateway reachable?** `ping -c4 <gateway>`
3. **Internet?** `ping -c4 1.1.1.1` — a stable IP.
4. **DNS?** `dig +short example.com` vs `dig @8.8.8.8 +short` — if only the second works, *your resolver* is broken.
5. **Path?** `traceroute example.com` — where does it die?
6. **Port + service?** `nc -zv host 443` from outside; on the server, **`ss -tunlp`** (the modern `netstat`) shows every listening TCP/UDP socket with its process.
7. **The wire?** `sudo tcpdump -i eth0 port 22` when it's still mysterious.

The pattern: **ping the host, test the port, resolve the name, trace the path, check the service is listening.** Know `ping`, `traceroute`, `ss -tunlp`, `dig`, `nc`, and `tcpdump`.
